In [1]:
import pandas as pd 
import os

In [2]:
df = pd.read_csv("wound_image_metadata.csv")

In [3]:
print("Total images:", len(df))
df.head()

Total images: 8129


,image_name,image_path,category,injury_type,severity
0,Female_normal-001.jpg,C:\Users\rekha\OneDrive\Desktop\New folder\fit...,normal,NaN,NaN
1,Female_normal-002.jpg,C:\Users\rekha\OneDrive\Desktop\New folder\fit...,normal,NaN,NaN
2,Female_normal-003.jpg,C:\Users\rekha\OneDrive\Desktop\New folder\fit...,normal,NaN,NaN
3,Female_normal-004.jpg,C:\Users\rekha\OneDrive\Desktop\New folder\fit...,normal,NaN,NaN
4,Female_normal-005.jpg,C:\Users\rekha\OneDrive\Desktop\New folder\fit...,normal,NaN,NaN


In [9]:
df.tail()

,image_name,image_path,category,injury_type,severity
8124,wound_mask-2682.jpg,C:\Users\rekha\OneDrive\Desktop\New folder\fit...,mask,NaN,NaN
8125,wound_mask-2683.jpg,C:\Users\rekha\OneDrive\Desktop\New folder\fit...,mask,NaN,NaN
8126,wound_mask-2684.jpg,C:\Users\rekha\OneDrive\Desktop\New folder\fit...,mask,NaN,NaN
8127,wound_mask-2685.jpg,C:\Users\rekha\OneDrive\Desktop\New folder\fit...,mask,NaN,NaN
8128,wound_mask-2686.jpg,C:\Users\rekha\OneDrive\Desktop\New folder\fit...,mask,NaN,NaN


In [4]:
df.columns

Index(['image_name', 'image_path', 'category', 'injury_type', 'severity'], dtype='str')

In [5]:
df["category"].value_counts()

category
normal    2757
wound     2686
mask      2686
Name: count, dtype: int64

In [7]:
df.isnull().sum()

image_name        0
image_path        0
category          0
injury_type    8129
severity       8129
dtype: int64

In [8]:
df_clean = df[df["category"] != "mask"]

print(df_clean["category"].value_counts())
print("Total images:", len(df_clean))

category
normal    2757
wound     2686
Name: count, dtype: int64
Total images: 5443


In [10]:
df_clean.to_csv("cleaned_image_metadata.csv", index=False)

In [11]:
df_clean["path_exists"] = df_clean["image_path"].apply(os.path.exists)
print(df_clean["path_exists"].value_counts())

path_exists
True    5443
Name: count, dtype: int64


In [13]:
from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(
    df_clean,test_size=0.30, random_state=42,stratify=df_clean["category"])

val_df, test_df = train_test_split(temp_df, test_size=0.50,random_state=42,stratify=temp_df["category"])

In [14]:
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Testing images:", len(test_df))

Training images: 3810
Validation images: 816
Testing images: 817


In [15]:
train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

In [16]:
import tensorflow as tf


In [17]:
print("Training:")
print(train_df["category"].value_counts())

print("\nValidation:")
print(val_df["category"].value_counts())

print("\nTesting:")
print(test_df["category"].value_counts())

Training:
category
normal    1930
wound     1880
Name: count, dtype: int64

Validation:
category
normal    413
wound     403
Name: count, dtype: int64

Testing:
category
normal    414
wound     403
Name: count, dtype: int64


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

preprocess = tf.keras.applications.mobilenet_v2.preprocess_input

train_datagen = ImageDataGenerator(preprocessing_function=preprocess)
val_datagen = ImageDataGenerator(preprocessing_function=preprocess)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess)

train_data = train_datagen.flow_from_dataframe(
    train_df,
    x_col="image_path",y_col="category",target_size=(224, 224),batch_size=32,class_mode="binary",classes=["normal", "wound"])

val_data = val_datagen.flow_from_dataframe(
    val_df,x_col="image_path",y_col="category",target_size=(224, 224),batch_size=32,class_mode="binary",classes=["normal", 
     "wound"],shuffle=False)

test_data = test_datagen.flow_from_dataframe(
    test_df,
    x_col="image_path",y_col="category",target_size=(224, 224), batch_size=32,class_mode="binary",classes=["normal","wound"],
    shuffle=False)

Found 3810 validated image filenames belonging to 2 classes.
Found 816 validated image filenames belonging to 2 classes.
Found 817 validated image filenames belonging to 2 classes.


In [19]:
print(train_data.class_indices)

{'normal': 0, 'wound': 1}


In [20]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [21]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)

output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 5s 1us/step


In [22]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.0001),loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [24]:
history = model.fit(train_data,validation_data=val_data,epochs=10)

Epoch 1/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 206s 2s/step - accuracy: 0.7003 - loss: 0.5650 - val_accuracy: 0.9142 - val_loss: 0.3458
Epoch 2/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 199s 2s/step - accuracy: 0.9058 - loss: 0.2948 - val_accuracy: 0.9706 - val_loss: 0.1995
Epoch 3/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 250s 2s/step - accuracy: 0.9609 - loss: 0.1825 - val_accuracy: 0.9877 - val_loss: 0.1360
Epoch 4/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 215s 2s/step - accuracy: 0.9772 - loss: 0.1311 - val_accuracy: 0.9890 - val_loss: 0.1016
Epoch 5/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 219s 2s/step - accuracy: 0.9819 - loss: 0.1044 - val_accuracy: 0.9926 - val_loss: 0.0801
Epoch 6/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 294s 2s/step - accuracy: 0.9877 - loss: 0.0834 - val_accuracy: 0.9963 - val_loss: 0.0656
Epoch 7/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 312s 2s/step - accuracy: 0.9900 - loss: 0.0700 - val_accuracy: 0.9975 - val_loss: 0.0554
Epoch 8/10
120/120 ━━━━━━━━━━━━━━━━━━━━ 215s 2s/step - accuracy: 0.9885 - loss: 0.0616 - val_accu

In [25]:
model.save("mobilenetv2_wound_model.keras")

In [26]:
test_loss, test_accuracy = model.evaluate(test_data)

print("Test Accuracy:", test_accuracy)
print("Test Loss:", test_loss)

26/26 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.9890 - loss: 0.0489
Test Accuracy: 0.9889841079711914
Test Loss: 0.04885757341980934


In [27]:
from sklearn.metrics import classification_report, confusion_matrix

test_data.reset()

y_true = test_data.classes
y_pred_prob = model.predict(test_data)
y_pred = (y_pred_prob.ravel() >= 0.5).astype(int)

print(classification_report(y_true, y_pred,target_names=["Normal", "Wound"]))

26/26 ━━━━━━━━━━━━━━━━━━━━ 47s 2s/step
              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99       414
       Wound       0.99      0.99      0.99       403

    accuracy                           0.99       817
   macro avg       0.99      0.99      0.99       817
weighted avg       0.99      0.99      0.99       817



In [28]:
print(confusion_matrix(y_true, y_pred))

[[409   5]
 [  4 399]]


In [30]:
model.save("mobilenetv2_wound_model.keras")

In [1]:
import tensorflow as tf

model = tf.keras.models.load_model(
    "mobilenetv2_wound_model.keras",
    compile=False
)

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

tflite_model = converter.convert()

with open("mobilenetv2_wound_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model converted successfully!")

INFO:tensorflow:Assets written to: C:\Users\rekha\AppData\Local\Temp\tmp6_1pzdx9\assets


INFO:tensorflow:Assets written to: C:\Users\rekha\AppData\Local\Temp\tmp6_1pzdx9\assets


Saved artifact at 'C:\Users\rekha\AppData\Local\Temp\tmp6_1pzdx9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2183889214672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2183889216400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2183889216208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2183889216976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2183889215248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2183889217360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2183889218128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2183889217936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2183889217168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2183889216592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  218388